# Podcast Words: Colab GPU und Drive

Das Notebook ruft dieselbe gesperrte Python-CLI wie Windows auf. Vor dem Start in Colab einen GPU-Laufzeittype wählen, das Repository mit `uv.lock` unter `/content/podcast_words_v3` bereitstellen und Drive verbinden. Es gibt keine automatische GPU-Zusage; GPU, Treiber, CUDA/ORT-Laufzeit und tatsächliche Knotenausführung werden getrennt erfasst.

Audio, decodierte WAV-Dateien und Modellcache liegen ausschließlich unter `/content` und sind nach Ende der Laufzeit nicht dauerhaft. Bei einer neuen Sitzung dasselbe Drive-Verzeichnis und dieselbe Folgenauswahl verwenden. Gültige Dateien werden durch `podcast-words process` übersprungen; fehlende oder ungültige Dateien werden erneut verarbeitet. Die GPU-Gruppe sperrt `onnxruntime-gpu==1.27.0`; ihre CUDA-13-/cuDNN-9-Bibliotheken müssen zur Colab-Laufzeit passen. Bei fehlendem CUDA-Provider stoppt das Notebook vor der Verarbeitung.


In [ ]:
import json
import platform
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

project = Path("/content/podcast_words_v3")
if not (project / "uv.lock").is_file():
    raise FileNotFoundError("Repository mit uv.lock unter /content/podcast_words_v3 bereitstellen.")
drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/podcast-words-v3")
drive_root.mkdir(parents=True, exist_ok=True)
pip_started = time.perf_counter()
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.10.9"], check=True)
pip_install_seconds = time.perf_counter() - pip_started
uv = [sys.executable, "-m", "uv"]
uv_sync_started = time.perf_counter()
subprocess.run(
    [*uv, "sync", "--locked", "--no-default-groups", "--group", "gpu", "--python", "3.12"],
    cwd=project,
    check=True,
)
uv_sync_seconds = time.perf_counter() - uv_sync_started
setup_measurements = {
    "pip_install_seconds": round(pip_install_seconds, 6),
    "uv_sync_seconds": round(uv_sync_seconds, 6),
    "total_dependency_setup_seconds": round(pip_install_seconds + uv_sync_seconds, 6),
}
cache_dir = Path("/content/podcast-words-cache")
work_dir = Path("/content/podcast-words-work")
cache_dir.mkdir(parents=True, exist_ok=True)
work_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import re

gpu_command = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_command.returncode != 0:
    raise RuntimeError(
        "Keine lesbare NVIDIA-GPU in dieser Laufzeit; es wurde keine Folge verarbeitet.\n"
        f"nvidia-smi: {gpu_command.stderr.strip()}"
    )
driver_match = re.search(r"Driver Version:\s*(\d+)", gpu_command.stdout)
if not driver_match:
    raise RuntimeError(
        "Die Treiberversion konnte nicht aus nvidia-smi gelesen werden; "
        "CUDA-Kompatibilit?t ist ungekl?rt und es wurde keine Folge verarbeitet."
    )
driver_major = int(driver_match.group(1))
if driver_major < 580:
    raise RuntimeError(
        f"Gefundener NVIDIA-Treiber {driver_major} ist zu alt f?r das "
        "gelockte ONNX Runtime 1.27 CUDA-13-Wheel (mindestens 580 erforderlich); "
        "es wurde keine Folge verarbeitet."
    )

gpu_python = "\n".join(
    [
        "import contextlib, importlib.metadata as metadata, io, json, onnxruntime, platform",
        "try:",
        "    onnxruntime.preload_dlls()",
        "    debug = io.StringIO()",
        "    with contextlib.redirect_stdout(debug):",
        "        onnxruntime.print_debug_info()",
        "except Exception as exc:",
        "    raise RuntimeError(f'CUDA/cuDNN DLL preflight failed: {type(exc).__name__}: {exc}') from exc",
        "print(json.dumps({'python': platform.python_version(), 'onnxruntime': onnxruntime.__version__, 'build_info': onnxruntime.get_build_info(), 'available_providers': onnxruntime.get_available_providers(), 'onnxruntime_gpu_distribution': metadata.version('onnxruntime-gpu'), 'cuda_library_debug_info': debug.getvalue()}))",
    ]
)
try:
    gpu_runtime = subprocess.run(
        [
            *uv,
            "run",
            "--frozen",
            "--no-default-groups",
            "--group",
            "gpu",
            "python",
            "-c",
            gpu_python,
        ],
        cwd=project,
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        "GPU-Runtime-Vorpr?fung fehlgeschlagen; ONNX Runtime konnte CUDA/cuDNN "
        "nicht initialisieren. Pr?fe Colab-Treiber, CUDA 13 und cuDNN 9. "
        "Es wurde keine Folge verarbeitet.\n"
        f"stdout:\n{exc.stdout}\nstderr:\n{exc.stderr}"
    ) from exc

runtime_facts = {
    "nvidia_smi": gpu_command.stdout,
    "nvidia_driver_major": driver_major,
    "onnxruntime": json.loads(gpu_runtime.stdout),
    "onnxruntime_gpu_distribution": json.loads(gpu_runtime.stdout)["onnxruntime_gpu_distribution"],
    "python": sys.version,
    "platform": platform.platform(),
}
if "CUDAExecutionProvider" not in runtime_facts["onnxruntime"]["available_providers"]:
    raise RuntimeError(
        "CUDAExecutionProvider fehlt in der installierten GPU-Laufzeit; keine Inferenz gestartet."
    )
print(json.dumps(runtime_facts, indent=2, ensure_ascii=False))
os.environ["PODCAST_WORDS_EXECUTION_PROVIDER"] = "CUDAExecutionProvider"
print(
    "CUDA ist angefordert; Provider-Verf?gbarkeit und DLL-Diagnose belegen "
    "keine ausgef?hrten GPU-Knoten. Erst die CLI-Profile weisen diese nach."
)

## Bereits gespeicherte ORT-Profile auswerten

Diese Zelle liest den vorhandenen Drive-Laufbericht und fasst dessen drei Rohprofile auf der VM zusammen. Sie startet keine Transkription, l?dt keine Profile auf den lokalen Rechner und ver?ndert weder den urspr?nglichen Laufbericht noch die Rohdateien. Die neue kompakte Zusammenfassung wird als separate Datei unter `reports/` auf Drive gespeichert. Sie enth?lt Profilbytegr??en, `Node`-Anzahlen und Host-Operatorzeiten je Anbieter, vorhandene `Kernel`-Ger?tezeiten, ?bertragungs-/Synchronisationsereignisse sowie den beobachteten Ereigniszeitraum.

Dauersummen der Host-`Node`-Ereignisse k?nnen sich ?ber Threads ?berlappen und sind keine Wandzeit. CUDA-Kernzeiten sind nur vorhanden, wenn das Profil `Kernel`-Ereignisse enth?lt; das h?ngt von CUPTI und dem ORT-Build ab.


In [ ]:
import hashlib
import shutil
import time

existing_episode_id = "ep-5764a1149f18a8f68698fb45bb405264"
existing_report_path = drive_root / "reports" / f"{existing_episode_id}.json"
if not existing_report_path.is_file():
    raise FileNotFoundError(f"Vorhandener Laufbericht fehlt: {existing_report_path}")
existing_report_bytes = existing_report_path.read_bytes()
existing_report = json.loads(existing_report_bytes)
existing_profiles = existing_report["process"]["execution_profile"]["drive_profile_files"]
summary_dir = work_dir / "profile-summaries"
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"{existing_episode_id}-profile-summary.json"
summary_command = [
    *uv,
    "run",
    "--frozen",
    "--no-default-groups",
    "--group",
    "gpu",
    "podcast-words",
    "profile-summary",
]
for session_name, profile_path in existing_profiles.items():
    if not Path(profile_path).is_file():
        raise FileNotFoundError(f"Drive-ORT-Profil fehlt: {profile_path}")
    summary_command.extend(["--profile", f"{session_name}={profile_path}"])
summary_command.extend(["--output", str(summary_path)])
summary_started = time.perf_counter()
summary_run = subprocess.run(
    summary_command, cwd=project, capture_output=True, text=True, check=True
)
summary_elapsed_seconds = time.perf_counter() - summary_started
summary_result = json.loads(summary_path.read_bytes())
episode_result_path = drive_root / "episodes" / f"{existing_episode_id}.json"
if not episode_result_path.is_file():
    raise FileNotFoundError(f"Gespeichertes Episodenergebnis fehlt: {episode_result_path}")
episode_result = json.loads(episode_result_path.read_bytes())
transcription = episode_result["transcription"]
profile_summary_evidence = {
    "source_report_path": str(existing_report_path),
    "source_report_sha256": hashlib.sha256(existing_report_bytes).hexdigest(),
    "summary_command_elapsed_seconds": round(summary_elapsed_seconds, 6),
    "source_runtime_facts": existing_report["runtime_facts"],
    "source_repository_revision": existing_report.get("repository_revision"),
    "episode_input_and_processing": {
        "audio_sha256": episode_result["audio"]["sha256"],
        "decoded_duration_seconds": episode_result["audio"]["duration_seconds"],
        "model_id": transcription["model_id"],
        "model_revision": transcription["model_revision"],
        "export_id": transcription["export_id"],
        "export_revision": transcription["export_revision"],
        "configuration_revision": transcription.get("configuration_revision"),
        "configuration": transcription["configuration"],
        "runtime_versions": transcription["runtime_versions"],
        "execution_providers": transcription["execution_providers"],
    },
    "profiles": summary_result["profiles"],
}
profile_summary_path = summary_dir / f"{existing_episode_id}-profile-evidence.json"
profile_summary_path.write_text(
    json.dumps(profile_summary_evidence, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
profile_summary_drive_path = drive_root / "reports" / profile_summary_path.name
summary_copy_started = time.perf_counter()
shutil.copy2(profile_summary_path, profile_summary_drive_path)
print(
    json.dumps(
        {
            "summary": str(profile_summary_drive_path),
            "source_report_sha256": profile_summary_evidence["source_report_sha256"],
            "sessions": len(profile_summary_evidence["profiles"]),
            "drive_copy_seconds": round(time.perf_counter() - summary_copy_started, 6),
        },
        indent=2,
    )
)

## Folgen verarbeiten oder in einer neuen Sitzung fortsetzen

Die erste voreingestellte Folge ist eine aktuelle Katalogkandidatin; sie ist kein Beleg für die endgültige Fünfer-Auswahl. Vor einem Pilotlauf die bestätigten stabilen IDs unten eintragen. Es wird jeweils genau eine Folge verarbeitet. Der nächste Lauf beginnt erst, nachdem Ergebnis, aktualisierter Katalog und Profilbericht auf Drive kopiert wurden.


In [ ]:
import os
import shutil
import tempfile

catalog_source = project / "data/podcast-ufo/catalog.json"
drive_catalog = drive_root / "catalog.json"
vm_catalog = work_dir / "catalog.json"
vm_episodes = work_dir / "episodes"
vm_episodes.mkdir(parents=True, exist_ok=True)
drive_episodes = drive_root / "episodes"
drive_reports = drive_root / "reports"
drive_profiles = drive_root / "profiles"
for directory in (drive_episodes, drive_reports, drive_profiles):
    directory.mkdir(parents=True, exist_ok=True)
if drive_catalog.is_file():
    shutil.copy2(drive_catalog, vm_catalog)
else:
    shutil.copy2(catalog_source, vm_catalog)


def atomic_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            dir=destination.parent, prefix=f".{destination.name}.", suffix=".partial", delete=False
        ) as stream:
            temporary = Path(stream.name)
            with Path(source).open("rb") as original:
                shutil.copyfileobj(original, stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, destination)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


episode_ids = ["ep-5764a1149f18a8f68698fb45bb405264"]
# Replace with the pilot IDs recorded in the candidate inventory before the five-episode run.
for episode_id in episode_ids:
    drive_result = drive_episodes / f"{episode_id}.json"
    vm_result = vm_episodes / f"{episode_id}.json"
    if drive_result.is_file():
        atomic_copy(drive_result, vm_result)
    command = [
        *uv,
        "run",
        "--frozen",
        "--no-default-groups",
        "--group",
        "gpu",
        "podcast-words",
        "process",
        "--catalog",
        str(vm_catalog),
        "--episode-id",
        episode_id,
        "--output-dir",
        str(vm_episodes),
        "--cache-dir",
        str(cache_dir),
    ]
    completed = subprocess.run(command, cwd=project, capture_output=True, text=True)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(
            f"process schlug für {episode_id} fehl (Exit {completed.returncode}); Teilstand bleibt auf Drive unverändert."
        )
    process_report = json.loads(completed.stdout)
    episode_status = process_report["results"][0]["status"]
    if episode_status == "completed":
        profile = process_report.get("execution_profile")
        if not profile or profile["model_node_events"].get("CUDAExecutionProvider", 0) < 1:
            raise RuntimeError(
                f"{episode_id}: kein tatsächlich ausgeführter CUDA-Modellknoten im ORT-Profil."
            )
        copied_profiles = {}
        for session_name, profile_path in profile["profile_files"].items():
            destination = drive_profiles / f"{episode_id}-{session_name}.json"
            atomic_copy(profile_path, destination)
            copied_profiles[session_name] = str(destination)
        profile["drive_profile_files"] = copied_profiles
        evidence = {"runtime_facts": runtime_facts, "process": process_report}
    elif episode_status == "skipped":
        previous_report = drive_reports / f"{episode_id}.json"
        if not previous_report.is_file():
            raise RuntimeError(
                f"{episode_id}: Ergebnis ist vorhanden, aber es fehlt das frühere ORT-Ausführungsprofil auf Drive; mit --retranscribe gezielt erneut verarbeiten."
            )
        evidence = json.loads(previous_report.read_text(encoding="utf-8"))
        old_profile = evidence["process"].get("execution_profile")
        if not old_profile or old_profile["model_node_events"].get("CUDAExecutionProvider", 0) < 1:
            raise RuntimeError(
                f"{episode_id}: gespeicherter Bericht belegt keine CUDA-Knotenausführung."
            )
        evidence["resume"] = {"status": "skipped", "new_session_runtime_facts": runtime_facts}
    else:
        raise RuntimeError(f"Unerwarteter Verarbeitungsstatus: {episode_status}")
    atomic_copy(vm_result, drive_result)
    atomic_copy(vm_catalog, drive_catalog)
    report_path = drive_reports / f"{episode_id}.json"
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="\n",
        dir=drive_reports,
        prefix=f".{episode_id}.",
        suffix=".partial",
        delete=False,
    ) as stream:
        temporary_report = Path(stream.name)
        json.dump(evidence, stream, indent=2, ensure_ascii=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary_report, report_path)
    print(f"{episode_id}: Folge und Laufbericht sind auf Drive gesichert.")

## Schwierige Ausschnitte und Zeitmarken

Für schwierige Stellen nach dem Abhören konkrete Zeitfenster festlegen und denselben gesperrten `podcast-words transcribe --audio-file ...`-Befehl auf VM-Ausschnitten starten. Diese Ausschnittausgaben bleiben getrennte Prüfartefakte und werden nicht in `episodes/` als vollständige Folge gespeichert. Die vollständige `process`-Ausgabe enthält Zeitmarken auf der Original-Audioachse; Anfang, mehrere Sprachabschnitte, Pausen, Mitte und Ende müssen gegen abgehörte Referenzstellen notiert werden.

Eine verfügbare CUDA-Session oder Providerliste ist kein GPU-Nachweis. `process` beendet die ORT-Profile nach der Erkennung, zählt ausgeführte `Node`-Ereignisse je Anbieter und schlägt fehl, wenn kein Modellknoten auf CUDA lief. CPU-Knoten bleiben separat sichtbar: quantisierte Operatoren können gemischt auf CPU laufen. Die Rohprofile werden pro Sitzung auf Drive unter `profiles/` gesichert.


In [ ]:
import hashlib
import urllib.request
import uuid

excerpt_windows = []  # Nach dem Abhören z. B. [(start_seconds, end_seconds)] eintragen.
if not excerpt_windows:
    print("Ausgelassen: Noch keine manuell ausgewählten schwierigen Ausschnitte eingetragen.")
else:
    catalog_data = json.loads(vm_catalog.read_bytes())
    catalog_entry = next(
        item for item in catalog_data["episodes"] if item["episode_id"] == episode_ids[0]
    )
    audio_url = catalog_entry["enclosures"][0]["url"]
    audio_file = cache_dir / "audio" / f"{hashlib.sha256(audio_url.encode()).hexdigest()}.source"
    audio_file.parent.mkdir(parents=True, exist_ok=True)
    if not audio_file.is_file():
        urllib.request.urlretrieve(audio_url, audio_file)
    metadata_file = work_dir / "excerpt-metadata.json"
    metadata_file.write_text(
        json.dumps(
            {
                "podcast_id": catalog_data["podcast_id"],
                "episode_id": catalog_entry["episode_id"],
                "source": catalog_entry["source"],
                "title": catalog_entry["title"],
                "published_at": catalog_entry["published_at"],
                "episode_number": catalog_entry.get("episode_number"),
                "webpage_url": catalog_entry.get("webpage_url"),
                "audio_url": audio_url,
            },
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    excerpt_evidence = []
    for index, (start_seconds, end_seconds) in enumerate(excerpt_windows, start=1):
        if not (
            0 <= start_seconds < end_seconds <= catalog_entry.get("duration_seconds", float("inf"))
        ):
            raise ValueError(f"Ungültiges Ausschnittfenster: {(start_seconds, end_seconds)}")
        clip_file = work_dir / f"{catalog_entry['episode_id']}-excerpt-{index}.wav"
        excerpt_file = (
            work_dir / f"{catalog_entry['episode_id']}-excerpt-{index}-{uuid.uuid4().hex}.json"
        )
        subprocess.run(
            [
                "ffmpeg",
                "-nostdin",
                "-v",
                "error",
                "-ss",
                str(start_seconds),
                "-i",
                str(audio_file),
                "-t",
                str(end_seconds - start_seconds),
                "-vn",
                "-ac",
                "1",
                "-ar",
                "16000",
                "-c:a",
                "pcm_s16le",
                str(clip_file),
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        excerpt_run = subprocess.run(
            [
                *uv,
                "run",
                "--frozen",
                "--no-default-groups",
                "--group",
                "gpu",
                "podcast-words",
                "transcribe",
                "--metadata",
                str(metadata_file),
                "--output",
                str(excerpt_file),
                "--audio-file",
                str(clip_file),
                "--cache-dir",
                str(cache_dir),
            ],
            cwd=project,
            capture_output=True,
            text=True,
        )
        print(excerpt_run.stdout)
        if excerpt_run.returncode != 0:
            raise RuntimeError(f"Ausschnitt {index} schlug fehl: {excerpt_run.stderr}")
        excerpt_run_report = json.loads(excerpt_run.stdout)
        profile = excerpt_run_report.get("execution_profile")
        if not profile or profile["model_node_events"].get("CUDAExecutionProvider", 0) < 1:
            raise RuntimeError(
                f"Ausschnitt {index}: ORT-Profil belegt keine CUDA-Modellknotenausführung."
            )
        copied_profiles = {}
        for session_name, profile_path in profile["profile_files"].items():
            destination = (
                drive_profiles
                / f"{catalog_entry['episode_id']}-excerpt-{index}-{session_name}.json"
            )
            atomic_copy(profile_path, destination)
            copied_profiles[session_name] = str(destination)
        excerpt_episode = json.loads(excerpt_file.read_bytes())
        excerpt_evidence.append(
            {
                "window_seconds": [start_seconds, end_seconds],
                "local_audio_sha256": excerpt_episode["audio"]["sha256"],
                "profile_model_node_events": profile["model_node_events"],
                "drive_profile_files": copied_profiles,
                "mapped_segments_on_episode_timeline": [
                    {
                        "text": segment["text"],
                        "start_seconds": start_seconds + segment["start_seconds"],
                        "end_seconds": start_seconds + segment["end_seconds"],
                        "token_points": [
                            {
                                "text": point["text"],
                                "at_seconds": start_seconds + point["at_seconds"],
                            }
                            for point in segment["token_points"]
                        ],
                    }
                    for segment in excerpt_episode["segments"]
                ],
            }
        )
    excerpt_dir = drive_root / "excerpts"
    excerpt_dir.mkdir(parents=True, exist_ok=True)
    (excerpt_dir / f"{catalog_entry['episode_id']}.json").write_text(
        json.dumps({"windows": excerpt_evidence}, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print(
        "Ausschnitte und versetzte Episodenzeiten getrennt von der vollständigen Folge auf Drive gespeichert."
    )

## Reproduzierbarer Laufzeitvergleich

Die folgende Zelle startet einen getrennten UFO537-Benchmark mit derselben Audio-SHA-256-Pr?fsumme wie der dokumentierte Windows-CPU-Lauf. Sie erfasst einen Lauf mit kaltem Audio-/Modellcache und tats?chlichem ORT-Profil, danach warme Wiederholungsl?ufe mit Profil an/aus. Jeder Lauf verwendet `--retranscribe`; ein vorhandenes Ergebnis kann keinen Resume-Skip vort?uschen. Folgen, Profile und Berichte landen getrennt unter `benchmarks/` auf VM und Drive. Profil-off-L?ufe weisen ausdr?cklich keinen Knotenausf?hrungsnachweis aus.

Die CLI misst Downloads, Sitzungsaufbau, Decode, Inferenz, Validierung/Serialisierung und Profilabschluss/-analyse. Der Notebookbericht misst Abh?ngigkeitsaufbau und Drive-Kopien. ORT-Knotenereignisse werden je Provider mit Anzahl und Ereignisdauersumme in Mikrosekunden erfasst; parallele Ereignisse k?nnen sich ?berlappen und ihre Summe ist keine Wandzeit. `snapshot_download` gibt keine verl?sslichen Netzwerkbytes an: Cachedatei-Bytes und Audio?bertragungsbytes bleiben getrennt. Der Bericht bindet Zweig, Commit, Audiohash und Laufzeit.


In [ ]:
import hashlib
import time

benchmark_started = time.perf_counter()
benchmark_root = work_dir / "runtime-benchmark"
benchmark_cache = benchmark_root / "cache"
benchmark_catalog = benchmark_root / "catalog.json"
benchmark_outputs = benchmark_root / "episodes"
benchmark_reports = benchmark_root / "reports"
for directory in (benchmark_outputs, benchmark_reports):
    directory.mkdir(parents=True, exist_ok=True)
shutil.copy2(vm_catalog, benchmark_catalog)
catalog_data = json.loads(benchmark_catalog.read_bytes())
episode_id = "ep-5764a1149f18a8f68698fb45bb405264"
expected_audio_sha256 = "d7f924479192f60229b5ba800227f640bb18a73527e360cc909f3657839327cf"
catalog_entry = next(item for item in catalog_data["episodes"] if item["episode_id"] == episode_id)
audio_url = catalog_entry["enclosures"][0]["url"]
audio_key = hashlib.sha256(audio_url.encode()).hexdigest()
audio_file = benchmark_cache / "audio" / f"{audio_key}.source"
benchmark_facts = {
    "runtime_facts": runtime_facts,
    "repository_branch": subprocess.run(
        ["git", "branch", "--show-current"], cwd=project, capture_output=True, text=True, check=True
    ).stdout.strip(),
    "repository_commit": subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=project, capture_output=True, text=True, check=True
    ).stdout.strip(),
    "setup_measurements": setup_measurements,
}
os.environ["PODCAST_WORDS_CODE_REVISION"] = benchmark_facts["repository_commit"]
evidence = {
    "schema_version": 1,
    "episode_id": episode_id,
    "expected_audio_sha256": expected_audio_sha256,
    "repository": benchmark_facts,
    "cache_policy": {
        "cold": "fresh benchmark cache",
        "warm": "same cache and exact source bytes; every timed CLI run uses --retranscribe",
        "model_wire_bytes": "unavailable from snapshot_download; materialized cache bytes are reported",
    },
    "runs": [],
    "drive_copies": [],
}


def benchmark_run(name, profile, cold=False):
    output_dir = benchmark_outputs / name
    report_path = benchmark_reports / f"{name}.json"
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        *uv,
        "run",
        "--frozen",
        "--no-default-groups",
        "--group",
        "gpu",
        "podcast-words",
        "process",
        "--catalog",
        str(benchmark_catalog),
        "--episode-id",
        episode_id,
        "--output-dir",
        str(output_dir),
        "--cache-dir",
        str(benchmark_cache),
        "--retranscribe",
        "--run-report",
        str(report_path),
    ]
    if not cold:
        command.extend(["--audio-file", str(audio_file)])
    command.append("--profile" if profile else "--no-profile")
    started = time.perf_counter()
    completed = subprocess.run(command, cwd=project, capture_output=True, text=True)
    wrapper_seconds = time.perf_counter() - started
    if completed.returncode:
        raise RuntimeError(
            f"{name} failed (exit {completed.returncode})\nstdout:\n{completed.stdout}\nstderr:\n{completed.stderr}"
        )
    process_report = json.loads(completed.stdout)
    measurements = json.loads(report_path.read_bytes())
    result_path = output_dir / f"{episode_id}.json"
    transcript = json.loads(result_path.read_bytes())
    provenance = {
        "audio_sha256": transcript["audio"]["sha256"],
        "decoded_duration_seconds": transcript["audio"]["duration_seconds"],
        "model_id": transcript["transcription"]["model_id"],
        "model_revision": transcript["transcription"]["model_revision"],
        "export_id": transcript["transcription"]["export_id"],
        "export_revision": transcript["transcription"]["export_revision"],
        "configuration_revision": transcript["transcription"].get("configuration_revision"),
        "execution_providers": transcript["transcription"]["execution_providers"],
        "runtime_versions": transcript["transcription"]["runtime_versions"],
    }
    if provenance["audio_sha256"] != expected_audio_sha256:
        raise RuntimeError(
            f"Audio mismatch: expected {expected_audio_sha256}, got {provenance['audio_sha256']}; comparison is invalid."
        )
    profile_report = process_report.get("execution_profile")
    if profile and (
        not profile_report
        or profile_report["model_node_events"].get("CUDAExecutionProvider", 0) < 1
    ):
        raise RuntimeError(f"{name} has no executed CUDA model node evidence")
    if not profile and measurements["profile"]["execution_proof"] != "unavailable-profile-disabled":
        raise RuntimeError(f"{name} did not label profile-off evidence as unavailable")
    record = {
        "name": name,
        "profile_enabled": profile,
        "cold_cache_run": cold,
        "wrapper_wall_seconds": round(wrapper_seconds, 6),
        "measurements": measurements,
        "execution_profile": profile_report,
        "transcript_provenance": provenance,
        "episode_output": str(result_path),
    }
    evidence["runs"].append(record)
    return record


cold_run = benchmark_run("cold-unprofiled", profile=False, cold=True)
if not audio_file.is_file():
    raise RuntimeError("cold run did not leave its exact audio in the isolated benchmark cache")
benchmark_run("warm-profiled", profile=True)
benchmark_run("warm-unprofiled", profile=False)

drive_benchmarks = drive_root / "benchmarks"
for directory in (
    drive_benchmarks / "episodes",
    drive_benchmarks / "reports",
    drive_benchmarks / "profiles",
):
    directory.mkdir(parents=True, exist_ok=True)
for record in evidence["runs"]:
    source = Path(record["episode_output"])
    target = drive_benchmarks / "episodes" / record["name"] / source.name
    started = time.perf_counter()
    atomic_copy(source, target)
    evidence["drive_copies"].append(
        {
            "kind": "validated_episode",
            "run": record["name"],
            "bytes": source.stat().st_size,
            "elapsed_seconds": round(time.perf_counter() - started, 6),
        }
    )
    record["drive_episode"] = str(target)
    profile = record["execution_profile"]
    if profile:
        copied = {}
        for session_name, profile_path in profile["profile_files"].items():
            source = Path(profile_path)
            target = drive_benchmarks / "profiles" / f"{record['name']}-{session_name}.json"
            started = time.perf_counter()
            atomic_copy(source, target)
            evidence["drive_copies"].append(
                {
                    "kind": "ort_profile",
                    "run": record["name"],
                    "session": session_name,
                    "bytes": source.stat().st_size,
                    "elapsed_seconds": round(time.perf_counter() - started, 6),
                }
            )
            copied[session_name] = str(target)
        record["drive_profiles"] = copied

evidence["total_notebook_benchmark_seconds"] = round(time.perf_counter() - benchmark_started, 6)
evidence["conclusion"] = (
    "Compare warm profile-on/off wall times only after matching audio hash, configuration revision, code commit and runtime. Profile-off has no node execution proof."
)
summary = benchmark_reports / f"{episode_id}-benchmark.json"
summary.write_text(json.dumps(evidence, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
summary_target = drive_benchmarks / "reports" / summary.name
started = time.perf_counter()
atomic_copy(summary, summary_target)
print(
    json.dumps(
        {
            "local_report": str(summary),
            "drive_report": str(summary_target),
            "report_drive_copy_seconds": round(time.perf_counter() - started, 6),
            "cold_unprofiled_wall_seconds": cold_run["wrapper_wall_seconds"],
            "timed_runs": len(evidence["runs"]),
        },
        indent=2,
    )
)

## Ergebnisse zur lokalen Verarbeitung übertragen

Drive enthält `catalog.json`, `episodes/<Folgen-ID>.json`, `reports/<Folgen-ID>.json` und die ORT-Rohprofile in `profiles/`. Den `episodes/`-Ordner für den lokalen Import aus Drive herunterladen und nach der Integration von Ticket 11 über den konfliktprüfenden Befehl übernehmen: `podcast-words import-results --catalog data/podcast-ufo/catalog.json --input-dir <heruntergeladener-episodes-ordner> --output-dir data/podcast-ufo/episodes`. Die lokale Katalogdatei bleibt die Identitätsquelle. Gleiche Ergebnisse werden übersprungen, Konflikte ersetzen keine vorhandenen Dateien.

Notebookcode und Fixturechecks belegen keine Colab-Ausführung. GPU-Typ, Treiber/CUDA/ORT-Fakten, Rohprofile, schwierige Ausschnitte, vollständige Folge, Pausen-/Zeitmarkenprüfung und neue Sitzung müssen aus einem echten Lauf in Drive-Bericht und späterer Evidenz festgehalten werden.
